In [ ]:
import sys
import subprocess

required = ["transformers", "datasets", "scipy", "pandas", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128 if device == "mps" else 64
max_length = 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)
df["len1_chars"] = df["sentence1"].astype(str).str.len().astype(np.int32)
df["len2_chars"] = df["sentence2"].astype(str).str.len().astype(np.int32)
df["combined_len_chars"] = (df["len1_chars"] + df["len2_chars"]).astype(np.int32)

q1 = float(df["combined_len_chars"].quantile(1/3))
q2 = float(df["combined_len_chars"].quantile(2/3))

def length_bucket(x):
    if x <= q1:
        return "short"
    if x <= q2:
        return "medium"
    return "long"

df["length_bucket"] = df["combined_len_chars"].map(length_bucket)

print({
    "num_examples": len(df),
    "columns": df.columns.tolist(),
    "bucket_thresholds_chars": {"short_max": q1, "medium_max": q2},
    "bucket_counts": df["length_bucket"].value_counts().sort_index().to_dict(),
})
print(df.head())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print({
    "loaded_model": model_name,
    "device": device,
    "hidden_size": int(getattr(model.config, "hidden_size", -1)),
})


In [ ]:
sentences1 = df["sentence1"].astype(str).tolist()
sentences2 = df["sentence2"].astype(str).tolist()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=batch_size):
    all_embeddings = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            emb = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            emb = F.normalize(emb, p=2, dim=1)
            all_embeddings.append(emb.detach().cpu())
    return torch.cat(all_embeddings, dim=0)

emb1 = encode_texts(sentences1)
emb2 = encode_texts(sentences2)

cosine_scores = torch.sum(emb1 * emb2, dim=1).numpy().astype(np.float32)
predicted_score_0_5 = ((cosine_scores + 1.0) * 2.5).clip(0.0, 5.0).astype(np.float32)

preview_df = pd.DataFrame({
    "sentence1": df["sentence1"].head(10).tolist(),
    "sentence2": df["sentence2"].head(10).tolist(),
    "label": df["label"].head(10).tolist(),
    "cosine_score": cosine_scores[:10],
    "predicted_score_0_5": predicted_score_0_5[:10],
    "length_bucket": df["length_bucket"].head(10).tolist(),
})
print(preview_df)


In [ ]:
labels = df["label"].to_numpy(dtype=np.float32)

results_df = df.copy()
results_df["cosine_score"] = cosine_scores
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["abs_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

overall_pearson_cosine = pearsonr(results_df["cosine_score"], results_df["label"]).statistic
overall_spearman_cosine = spearmanr(results_df["cosine_score"], results_df["label"]).statistic
overall_pearson_rescaled = pearsonr(results_df["predicted_score_0_5"], results_df["label"]).statistic
overall_spearman_rescaled = spearmanr(results_df["predicted_score_0_5"], results_df["label"]).statistic
overall_mae = results_df["abs_error"].mean()

bucket_rows = []
for bucket_name, g in results_df.groupby("length_bucket", sort=True):
    bucket_rows.append({
        "length_bucket": bucket_name,
        "count": int(len(g)),
        "combined_len_chars_min": int(g["combined_len_chars"].min()),
        "combined_len_chars_max": int(g["combined_len_chars"].max()),
        "pearson": float(pearsonr(g["predicted_score_0_5"], g["label"]).statistic),
        "spearman": float(spearmanr(g["predicted_score_0_5"], g["label"]).statistic),
        "mae": float(np.mean(np.abs(g["predicted_score_0_5"] - g["label"]))),
        "label_mean": float(g["label"].mean()),
        "prediction_mean": float(g["predicted_score_0_5"].mean()),
    })

bucket_df = pd.DataFrame(bucket_rows).sort_values("length_bucket").reset_index(drop=True)

print("overall_metrics")
print({
    "pearson_cosine": float(overall_pearson_cosine),
    "spearman_cosine": float(overall_spearman_cosine),
    "pearson_rescaled_0_5": float(overall_pearson_rescaled),
    "spearman_rescaled_0_5": float(overall_spearman_rescaled),
    "mae_rescaled_0_5": float(overall_mae),
})

print("per_bucket_metrics")
print(bucket_df.to_string(index=False))

print("sample_results")
print(results_df[[
    "sentence1", "sentence2", "label", "cosine_score",
    "predicted_score_0_5", "length_bucket", "abs_error"
]].head(10).to_string(index=False))


In [ ]:
worst_examples = results_df[[
    "sentence1", "sentence2", "label", "cosine_score",
    "predicted_score_0_5", "length_bucket", "combined_len_chars", "abs_error"
]].sort_values("abs_error", ascending=False).head(10)

print("worst_examples_by_abs_error")
print(worst_examples.to_string(index=False))

runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"length_bucket_short_max_chars: {q1:.2f}")
print(f"length_bucket_medium_max_chars: {q2:.2f}")
print(f"pearson_cosine_score: {overall_pearson_cosine:.6f}")
print(f"spearman_cosine_score: {overall_spearman_cosine:.6f}")
print(f"pearson_rescaled_score_0_5: {overall_pearson_rescaled:.6f}")
print(f"spearman_rescaled_score_0_5: {overall_spearman_rescaled:.6f}")
print(f"mae_rescaled_score_0_5: {overall_mae:.6f}")
for _, row in bucket_df.iterrows():
    name = row["length_bucket"]
    print(f"bucket_{name}_count: {int(row['count'])}")
    print(f"bucket_{name}_pearson: {row['pearson']:.6f}")
    print(f"bucket_{name}_spearman: {row['spearman']:.6f}")
    print(f"bucket_{name}_mae: {row['mae']:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
